In [11]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# Pulling in necessary files and setting model parameters
CSV_PATH   = "paleoexfiltrationperm13meanfinal.csv"
OUT_FIG    = "paleo_persistence_all_sites.png"
OUT_TABLE  = "paleo_site_summary.csv"
DT_YR      = 2.0      # model time step (yr)
MOD_MAX    = 516.0    # modern ensemble maximum at k = 1e-13 m2 (Table 2.8), mm/yr
MOD_WINDOW = 16.0     # modern forcing window, 2003-2019 (yr)
FRACS      = (0.25, 0.50)   # persistence thresholds, as fraction of each site's own peak

# Loading in files
df = pd.read_csv(CSV_PATH)
df.columns = ["site", "age", "lat", "lon", "q"]
g = df.groupby("site")

# Site metrics
peak = g.q.max()
peak_age = df.loc[g.q.idxmax()].set_index("site").age
end_q = df[df.age == df.age.min()].set_index("site").q          # rate at 0 ka
dur = {f: g.apply(lambda d, f=f: (d.q >= f * d.q.max()).sum() * DT_YR) for f in FRACS}
above_mod = g.q.apply(lambda x: (x >= MOD_MAX).sum() * DT_YR)   # yr above modern max

feat = peak.idxmax()                                             # highest-peak site
med = (dur[0.50] - dur[0.50].median()).abs().idxmin()            # median-persistence site

# Figure
fig = plt.figure(figsize=(12, 7), constrained_layout=True)
gs = GridSpec(2, 2, figure=fig, height_ratios=[1.25, 1])
a = fig.add_subplot(gs[0, :]); b = fig.add_subplot(gs[1, 0]); c = fig.add_subplot(gs[1, 1])

# (a) all sites
for _, d in g:
    a.plot(d.age / 1000, d.q, color="0.8", lw=0.6)
for s, col, lab in [(med, "C0", f"Site AV-{med} (median persistence)"),
                    (feat, "C3", f"Site M-{feat} (highest maximum)")]:
    d = df[df.site == s]
    a.plot(d.age / 1000, d.q, color=col, lw=2.5, label=lab)
a.axhline(MOD_MAX, color="k", ls="--", lw=1, label=f"Modern ensemble maximum ({MOD_MAX:.0f} mm yr$^{-1}$)")
a.axhline(0, color="k", lw=0.6)
# a.annotate("Model starts with no prior\nloading memory (rise from zero\nis partly an initialization effect)",
#            xy=(19.6, 60), xytext=(16.3, -70), fontsize=9, color="0.25",
#            arrowprops=dict(arrowstyle="->", color="0.4"))
a.set_ylim(-130, 850); a.invert_xaxis()
a.set_xlabel("Age (ka)"); a.set_ylabel("Exfiltration rate (mm yr$^{-1}$)")
a.set_title("(a) Paleo-exfiltration at all 60 sites (posterior mean)")
a.legend(frameon=False, loc="upper right")

# (b) persistence CDF
for f, col in zip(FRACS, ["C1", "C0"]):
    x = np.sort(np.clip(dur[f], 1, None) / 1000)
    b.step(x, np.arange(1, len(x) + 1) / len(x), color=col, lw=2.5, where="post",
           label=f"≥ {int(f*100)}% of site peak")
b.axvspan(1e-2, MOD_WINDOW / 1000, color="0.7", alpha=0.6, label="Modern forcing window (16 yr)")
b.set_xscale("log"); b.set_xlim(1e-2, 25)
b.set_xlabel("Time above threshold (kyr)"); b.set_ylabel("Fraction of sites")
b.set_title("(b) Persistence across sites"); b.legend(frameon=False, loc="upper left")

# (c) sites exceeding modern maximum
top = above_mod[above_mod > 0].sort_values(ascending=False)
c.bar([str(i) for i in top.index], top.values, color="C3")
c.set_yscale("log"); c.set_ylim(10, 4000)
c.axhline(MOD_WINDOW, color="0.35", ls=":", lw=1.5, label="Modern forcing window (16 yr)")
c.set_xlabel("Site"); c.set_ylabel(f"Years above {MOD_MAX:.0f} mm/yr$^{-1}$)")
c.set_title(f"(c) Time above modern maximum ({len(top)} sites)"); c.legend(frameon=False, loc="upper right")
fig.savefig(OUT_FIG, dpi=300)

# Table
out = pd.DataFrame({
    "peak_mm_yr": peak, "peak_age_yr": peak_age, "mean_mm_yr": g.q.mean(), "min_mm_yr": g.q.min(),
    "rate_at_0ka_mm_yr": end_q,
    "yr_above_25pct_peak": dur[0.25], "yr_above_50pct_peak": dur[0.50],
    "yr_above_modern_max": above_mod, "lat": g.lat.first(), "lon": g.lon.first()}).round(2)
out.to_csv(OUT_TABLE)

# Printing values
q = lambda s: s.quantile([.25, .75]).values
print(f"rows {len(df)}, sites {df.site.nunique()}, steps/site {g.size().unique()}")
print(f"overall: max {df.q.max():.1f} | min {df.q.min():.1f} | mean {df.q.mean():.1f} mm yr$^{-1}$")
print(f"sites with peak > {MOD_MAX:.0f}: {(peak > MOD_MAX).sum()}  -> {sorted(top.index.tolist())}")
print(f"years above modern max (those sites): {top.min():.0f}-{top.max():.0f}")
print(f"peak age: median {peak_age.median()/1000:.1f} ka, IQR {q(peak_age)/1000}; peaks before 15 ka: {(peak_age > 15000).sum()}")
for f in FRACS:
    d = dur[f]
    print(f"{int(f*100)}% of peak: median {d.median()/1000:.1f} kyr, IQR {q(d)/1000}, "
          f"min {d.min():.0f} yr, max {d.max()/1000:.1f} kyr, >=1 kyr: {(d>=1000).sum()}/60, >=5 kyr: {(d>=5000).sum()}/60")
print(f"median 50%-persistence / modern window = {dur[0.50].median()/MOD_WINDOW:.0f}x; all sites > window: {(dur[0.50] > MOD_WINDOW).all()}")
print(f"rate at 0 ka: all >0: {(end_q > 0).all()}, range {end_q.min():.1f}-{end_q.max():.1f}, median {end_q.median():.0f}; "
      f"median fraction of peak {(end_q/peak).median():.2f}")
print(f"sites with any infiltration: {(g.q.min() < 0).sum()}")
print(f"featured site {feat}, median-persistence site {med}")

rows 600060, sites 60, steps/site [10001]
overall: max 790.6 | min -81.7 | mean 116.7 mm yr$^-1$
sites with peak > 516: 8  -> [4, 40, 57, 62, 67, 125, 129, 156]
years above modern max (those sites): 98-1836
peak age: median 6.1 ka, IQR [4.104 7.879]; peaks before 15 ka: 2
25% of peak: median 12.4 kyr, IQR [ 8.1565 18.1095], min 1652 yr, max 19.9 kyr, >=1 kyr: 60/60, >=5 kyr: 55/60
50% of peak: median 6.3 kyr, IQR [ 2.947 10.501], min 270 yr, max 17.6 kyr, >=1 kyr: 53/60, >=5 kyr: 36/60
median 50%-persistence / modern window = 392x; all sites > window: True
rate at 0 ka: all >0: True, range 32.6-337.2, median 142; median fraction of peak 0.49
sites with any infiltration: 9
featured site 62, median-persistence site 45


In [5]:
import numpy as np
import pandas as pd

df = pd.read_csv("paleoexfiltrationperm13meanfinal.csv")
df.columns = ["site", "age", "lat", "lon", "q"]
g = df.groupby("site")
DT_YR = 2.0

# Rates scale as sqrt(k / 1e-13) (Eq. 2.3), so other permeabilities are rescalings
perms = [1e-11, 1e-12, 1e-13, 1e-14, 1e-15]
thresholds = [1, 10, 50, 100]          # mm/yr

rows = []
for k in perms:
    scale = np.sqrt(k / 1e-13)
    for thr in thresholds:
        d = g.q.apply(lambda x: (x * scale >= thr).sum() * DT_YR)
        rows.append({"k (m2)": f"{k:.0e}", "threshold (mm/yr)": thr,
                     "median (kyr)": d.median() / 1000,
                     "IQR low (kyr)": d.quantile(.25) / 1000,
                     "IQR high (kyr)": d.quantile(.75) / 1000,
                     "sites >= 15 kyr": int((d >= 15000).sum())})
table = pd.DataFrame(rows).round(2)
print(table.to_string(index=False))
table.to_csv("threshold_sensitivity.csv", index=False)

k (m2)  threshold (mm/yr)  median (kyr)  IQR low (kyr)  IQR high (kyr)  sites >= 15 kyr
 1e-11                  1         20.00          20.00           20.00               52
 1e-11                 10         20.00          19.85           20.00               51
 1e-11                 50         19.96          17.76           19.99               48
 1e-11                100         19.84          13.74           19.98               42
 1e-12                  1         20.00          19.99           20.00               52
 1e-12                 10         19.98          18.84           20.00               49
 1e-12                 50         19.59          12.02           19.94               38
 1e-12                100         18.18           8.42           19.75               33
 1e-13                  1         20.00          19.85           20.00               51
 1e-13                 10         19.84          13.74           19.98               42
 1e-13                 50       